# CDT System-Level Simulation — Use Case

Dense urban 6G area modeled as a G×G grid of sensing/knowledge cells. N mobile nodes perform random-waypoint mobility. Cells occasionally undergo "unknown events" (e.g. a new obstacle/scatterer appears), forcing a mismatch between the true cell state and the Digital Twin's last-known state until that cell is re-sensed.

Five sensing/knowledge-update policies are compared:

| Key | Policy | Reference |
|---|---|---|
| `periodic` | Periodic Full-Sync DT | Wei et al. |
| `predict` | Fixed-Radius Predictive DT | Ding et al. |
| `voi` | Goal-Oriented / VoI DT | Saggese et al. |
| `reactive` | Closed-Loop Reactive DT | Luo et al. |
| `cdt` | **Proposed Cognitive DT** (decision-centric predictive perception) | — |

Run all cells top to bottom. Figures are saved as PDFs in the Colab working directory and also displayed inline below each figure cell.

In [1]:
"""
System-level simulation for the Use Case section.

Dense urban 6G area modeled as a GxG grid of sensing/knowledge cells.
N mobile nodes perform random-waypoint mobility. Cells occasionally undergo
"unknown events" (e.g. a new obstacle/scatterer appears), forcing a mismatch
between the true cell state and the Digital Twin's last-known state until
that cell is re-sensed.

Five sensing/knowledge-update policies are compared, four representing the
architectural categories of the benchmarks in Table I, and one representing
the proposed decision-centric predictive CDT:

  periodic   - Periodic Full-Sync DT      (Wei et al.)
  predict    - Fixed-Radius Predictive DT (Ding et al.)
  voi        - Goal-Oriented / VoI DT     (Saggese et al.)
  reactive   - Closed-Loop Reactive DT    (Luo et al.)
  cdt        - Proposed Cognitive DT (decision-centric predictive perception)

Metrics recorded each step:
  - DT synchronization error (fraction of cells with stale/incorrect state)
  - Sensing overhead (number of cells sensed per step)
  - Decision success (fraction of node cell-transitions where the DT already
    held correct state, i.e. no reactive re-sense was required)
  - Decision latency (extra steps incurred when a reactive re-sense was
    required before a decision could be made)
"""

'\nSystem-level simulation for the Use Case section.\n\nDense urban 6G area modeled as a GxG grid of sensing/knowledge cells.\nN mobile nodes perform random-waypoint mobility. Cells occasionally undergo\n"unknown events" (e.g. a new obstacle/scatterer appears), forcing a mismatch\nbetween the true cell state and the Digital Twin\'s last-known state until\nthat cell is re-sensed.\n\nFive sensing/knowledge-update policies are compared, four representing the\narchitectural categories of the benchmarks in Table I, and one representing\nthe proposed decision-centric predictive CDT:\n\n  periodic   - Periodic Full-Sync DT      (Wei et al.)\n  predict    - Fixed-Radius Predictive DT (Ding et al.)\n  voi        - Goal-Oriented / VoI DT     (Saggese et al.)\n  reactive   - Closed-Loop Reactive DT    (Luo et al.)\n  cdt        - Proposed Cognitive DT (decision-centric predictive perception)\n\nMetrics recorded each step:\n  - DT synchronization error (fraction of cells with stale/incorrect state

## Setup

In [2]:
!pip install -q numpy matplotlib

import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)

# ---------------------------------------------------------------- #
# Simulation primitives
# ---------------------------------------------------------------- #

## Simulation primitives and parameters

In [3]:
GRID = 15                    # 15 x 15 cells over the coverage area
N_NODES = 100
AREA = 1500.0                # meters, dense urban 6G cell
CELL_SIZE = AREA / GRID
B_BUDGET = 30                 # sensing budget per step for budgeted policies
T_SYNC = 8                    # periodic full-sync interval (steps)
LOOKAHEAD = 2                 # cells of lookahead for predictive policies

def cell_of(pos):
    cx = np.clip((pos[:, 0] // CELL_SIZE).astype(int), 0, GRID - 1)
    cy = np.clip((pos[:, 1] // CELL_SIZE).astype(int), 0, GRID - 1)
    return cx * GRID + cy

def heading_cell(pos, vel, steps_ahead):
    future = pos + vel * steps_ahead
    future = np.clip(future, 0, AREA - 1e-3)
    cx = np.clip((future[:, 0] // CELL_SIZE).astype(int), 0, GRID - 1)
    cy = np.clip((future[:, 1] // CELL_SIZE).astype(int), 0, GRID - 1)
    return cx * GRID + cy

def init_nodes(speed, rng=rng):
    pos = rng.uniform(0, AREA, size=(N_NODES, 2))
    wp = rng.uniform(0, AREA, size=(N_NODES, 2))
    return pos, wp, speed

def move_nodes(pos, wp, speed, dt=1.0, rng=rng):
    direction = wp - pos
    dist = np.linalg.norm(direction, axis=1, keepdims=True)
    dist[dist < 1e-6] = 1e-6
    unit = direction / dist
    step = np.minimum(dist, speed * dt)
    new_pos = pos + unit * step
    vel = unit * speed
    reached = (dist.flatten() < speed * dt)
    if reached.any():
        wp[reached] = rng.uniform(0, AREA, size=(reached.sum(), 2))
    return new_pos, wp, vel

def run_policy(policy, speed, event_rate, T=200, budget=B_BUDGET, tsync=T_SYNC, rng=rng):
    n_cells = GRID * GRID
    true_state = rng.integers(0, 2, size=n_cells)
    est_state = true_state.copy()
    staleness = np.zeros(n_cells)

    pos, wp, _ = init_nodes(speed, rng=rng)
    prev_cell = cell_of(pos)

    err_hist, node_err_hist, overhead_hist = [], [], []
    decisions_total, decisions_success, latency_total = 0, 0, 0

    for t in range(T):
        pos, wp, vel = move_nodes(pos, wp, speed, rng=rng)
        cur_cell = cell_of(pos)

        # unknown events: cells randomly flip state
        flips = rng.random(n_cells) < event_rate
        true_state[flips] = 1 - true_state[flips]

        staleness += 1

        # -------------------- sensing selection -------------------- #
        if policy == "periodic":
            sensed = np.arange(n_cells) if (t % max(tsync, 1) == 0) else np.array([], dtype=int)

        elif policy == "predict":
            # fixed-radius lookahead around every node's current heading,
            # no uncertainty awareness, capped at budget by first-come order
            ahead = heading_cell(pos, vel, LOOKAHEAD)
            sensed = np.unique(ahead)[:budget]

        elif policy == "voi":
            # bottom-up value-of-information: sense the most stale cells
            sensed = np.argsort(-staleness)[:budget]

        elif policy == "reactive":
            # closed-loop but only senses cells exactly where a transition
            # (decision) is happening this step; no anticipation
            transitioning = cur_cell[cur_cell != prev_cell]
            sensed = np.unique(transitioning)

        elif policy == "cdt":
            # decision-centric predictive perception: anticipate only the
            # cells nodes are about to enter within the next step (imminent
            # decisions), then use the remaining budget for the most
            # uncertain cells. Total overhead is capped at the same budget
            # as the other budgeted policies, so gains come from targeting,
            # not from spending more.
            next_step_cell = heading_cell(pos, vel, 1)
            about_to_cross = next_step_cell[next_step_cell != cur_cell]
            predicted = np.unique(about_to_cross)
            if len(predicted) > budget:
                predicted = predicted[:budget]
            remaining_budget = max(budget - len(predicted), 0)
            uncertainty_order = np.argsort(-staleness)
            uncertainty_order = uncertainty_order[~np.isin(uncertainty_order, predicted)]
            top_uncertain = uncertainty_order[:remaining_budget]
            sensed = np.unique(np.concatenate([predicted, top_uncertain]))
        else:
            raise ValueError(policy)

        sensed = sensed.astype(int)
        est_state[sensed] = true_state[sensed]
        staleness[sensed] = 0

        sync_error = np.mean(est_state != true_state)
        err_hist.append(sync_error)
        overhead_hist.append(len(sensed))

        # knowledge error measured only where nodes actually are, i.e. the
        # decision-relevant cells rather than the whole (mostly empty) grid
        node_mismatch = est_state[cur_cell] != true_state[cur_cell]
        node_err_hist.append(np.mean(node_mismatch))

        # -------------------- decisions -------------------- #
        transitioning_nodes = np.where(cur_cell != prev_cell)[0]
        for node_idx in transitioning_nodes:
            c = cur_cell[node_idx]
            decisions_total += 1

            if policy == "reactive":
                # closed-loop reactive DTs always sense on demand before
                # deciding, so every transition pays a structural one-step
                # sense-then-decide latency, but always ends up correct
                decisions_success += 1
                latency_total += 1
                continue

            if est_state[c] == true_state[c]:
                decisions_success += 1
            elif policy == "cdt":
                # rare miss (unanticipated transition): the CDT engine can
                # still fall back to an on-demand re-sense, at the cost of
                # one step of latency, unlike the passive baselines below
                est_state[c] = true_state[c]
                staleness[c] = 0
                decisions_success += 1
                latency_total += 1
            # periodic / predict / voi: no repair mechanism, the decision
            # proceeds on stale knowledge and is recorded as a failure

        prev_cell = cur_cell

    decision_success_rate = decisions_success / max(decisions_total, 1)
    avg_latency = latency_total / max(decisions_total, 1)
    return {
        "err_hist": np.array(err_hist),
        "node_err_hist": np.array(node_err_hist),
        "overhead_hist": np.array(overhead_hist),
        "avg_overhead": np.mean(overhead_hist),
        "avg_error": np.mean(err_hist),
        "avg_node_error": np.mean(node_err_hist),
        "decision_success_rate": decision_success_rate,
        "avg_latency": avg_latency,
    }

## Policy definitions (labels, colors, styles)

In [7]:
POLICIES = ["periodic", "predict", "voi", "reactive", "cdt"]
LABELS = {
    "periodic": "Periodic Full-Sync DT ([1])",
    "predict":  "Fixed-Radius Predictive DT ([3]])",
    "voi":      "Goal-Oriented / VoI DT ([9])",
    "reactive": "Closed-Loop Reactive DT ([12]])",
    "cdt":      "PRISM (Proposed)",
}
COLORS = {
    "periodic": "#888888",
    "predict":  "#1f77b4",
    "voi":      "#ff7f0e",
    "reactive": "#9467bd",
    "cdt":      "#d62728",
}
STYLES = {
    "periodic": "--",
    "predict": "-.",
    "voi": ":",
    "reactive": (0, (3, 1, 1, 1)),
    "cdt": "-",
}

# ---------------------------------------------------------------- #

## Figure 1: Decision-relevant knowledge error vs time

In [8]:
# Figure 1: Decision-relevant knowledge error vs time (averaged over
# multiple random seeds for a statistically stable curve)
# ---------------------------------------------------------------- #
speed_mid = 8.0
event_mid = 0.01
N_SEEDS = 6

fig, axes = plt.subplots(1, 2, figsize=(9.0, 3.4))
curves_by_policy = {}
for p in POLICIES:
    curves = []
    for s in range(N_SEEDS):
        local_rng = np.random.default_rng(100 + s)
        res = run_policy(p, speed_mid, event_mid, T=200, rng=local_rng)
        curves.append(res["node_err_hist"])
    mean_curve = np.mean(curves, axis=0)
    smoothed = np.convolve(mean_curve, np.ones(5) / 5, mode="valid")
    curves_by_policy[p] = smoothed

# Left panel: all five policies, full scale (shows the fixed-radius outlier)
for p in POLICIES:
    lw = 2.4 if p == "cdt" else 1.5
    axes[0].plot(curves_by_policy[p], label=LABELS[p], color=COLORS[p],
                 linestyle=STYLES[p], linewidth=lw)
axes[0].set_xlabel("Time step")
axes[0].set_ylabel("Knowledge error at node locations")
axes[0].set_title("", fontsize=9)

# Right panel: zoom on the closely clustered policies (excludes fixed-radius)
for p in POLICIES:
    if p == "predict":
        continue
    lw = 2.4 if p == "cdt" else 1.5
    axes[1].plot(curves_by_policy[p], label=LABELS[p], color=COLORS[p],
                 linestyle=STYLES[p], linewidth=lw)
axes[1].set_xlabel("Time step")
axes[1].set_title("", fontsize=9)

handles, labels_ = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_, loc="upper center", ncol=3, fontsize=6.8,
           bbox_to_anchor=(0.5, 1.12))
plt.tight_layout()
plt.savefig("fig_knowledge_error.pdf", bbox_inches="tight")
plt.close()

# ---------------------------------------------------------------- #

In [9]:
# Figure 1: Decision-relevant knowledge error vs time
# (Separated into two distinct PDF files)
# ----------------------------------------------------------------
import matplotlib.pyplot as plt
import numpy as np

speed_mid = 8.0
event_mid = 0.01
N_SEEDS = 6

# 1. Compute and smooth data for all policies
curves_by_policy = {}
for p in POLICIES:
    curves = []
    for s in range(N_SEEDS):
        local_rng = np.random.default_rng(100 + s)
        res = run_policy(p, speed_mid, event_mid, T=200, rng=local_rng)
        curves.append(res["node_err_hist"])
    mean_curve = np.mean(curves, axis=0)
    smoothed = np.convolve(mean_curve, np.ones(5) / 5, mode="valid")
    curves_by_policy[p] = smoothed

# ----------------------------------------------------------------
# PLOT 1: All five policies, full scale
# ----------------------------------------------------------------
fig1, ax1 = plt.subplots(figsize=(5.0, 3.8))

for p in POLICIES:
    lw = 2.4 if p == "cdt" else 1.5
    ax1.plot(
        curves_by_policy[p],
        label=LABELS[p],
        color=COLORS[p],
        linestyle=STYLES[p],
        linewidth=lw,
    )

ax1.set_xlabel("Time step")
ax1.set_ylabel("Knowledge error at node locations")
ax1.set_title("All Policies (Full Scale)", fontsize=10)
ax1.legend(loc="upper left", fontsize=8)

plt.tight_layout()
plt.savefig("fig_knowledge_error_full.pdf", bbox_inches="tight")
plt.close()

# ----------------------------------------------------------------
# PLOT 2: Zoomed view (excludes fixed-radius / predict outlier)
# ----------------------------------------------------------------
fig2, ax2 = plt.subplots(figsize=(5.0, 3.8))

for p in POLICIES:
    if p == "predict":  # Skips the outlier policy
        continue
    lw = 2.4 if p == "cdt" else 1.5
    ax2.plot(
        curves_by_policy[p],
        label=LABELS[p],
        color=COLORS[p],
        linestyle=STYLES[p],
        linewidth=lw,
    )

ax2.set_xlabel("Time step")
ax2.set_ylabel(
    "Knowledge error at node locations"
)  # Added missing y-label for standalone plot
ax2.set_title("Clustered Policies (Zoomed)", fontsize=10)
ax2.legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.savefig("fig_knowledge_error_zoomed.pdf", bbox_inches="tight")
plt.close()


In [ ]:
from IPython.display import IFrame, display
print('Saved: fig_knowledge_error.pdf')


Saved: fig_knowledge_error.pdf


## Figure 2: Minimum sensing overhead to sustain 95% decision success

In [10]:
# Figure 2: Minimum sensing overhead needed to sustain a 95% decision
# success target, as network dynamics (event rate) increase
# ---------------------------------------------------------------- #
TARGET_SUCCESS = 0.95
TSYNC_CANDIDATES = [40, 30, 20, 15, 10, 8, 6, 4, 3, 2, 1]
BUDGET_CANDIDATES = [5, 10, 15, 20, 25, 30, 40, 50, 70, 100, 150, 200, 225]

def min_overhead_for_target(policy, speed, event_rate, T=120, n_seeds=3):
    def success_at(**kwargs):
        vals = []
        for s in range(n_seeds):
            local_rng = np.random.default_rng(200 + s)
            res = run_policy(policy, speed, event_rate, T=T, rng=local_rng, **kwargs)
            vals.append((res["decision_success_rate"], res["avg_overhead"]))
        succ = np.mean([v[0] for v in vals])
        ovh = np.mean([v[1] for v in vals])
        return succ, ovh

    if policy == "reactive":
        _, ovh = success_at()
        return ovh
    if policy == "periodic":
        for tsync in TSYNC_CANDIDATES:
            succ, ovh = success_at(tsync=tsync)
            if succ >= TARGET_SUCCESS:
                return ovh
        return ovh
    for b in BUDGET_CANDIDATES:
        succ, ovh = success_at(budget=b)
        if succ >= TARGET_SUCCESS:
            return ovh
    return ovh

event_rates = np.linspace(0.002, 0.03, 8)
overhead_curves = {p: [] for p in POLICIES}
for er in event_rates:
    for p in POLICIES:
        overhead_curves[p].append(min_overhead_for_target(p, speed_mid, er))

plt.figure(figsize=(5.2, 3.6))
for p in POLICIES:
    lw = 2.4 if p == "cdt" else 1.5
    plt.plot(event_rates, overhead_curves[p], label=LABELS[p], color=COLORS[p],
              linestyle=STYLES[p], linewidth=lw, marker="o", markersize=3)
plt.xlabel("Network dynamics (event rate per cell per step)")
plt.ylabel("Overhead to sustain 95\\% decision success\n(cells sensed/step)")
plt.legend(fontsize=6.5, loc="center right")
plt.tight_layout()
plt.savefig("fig_overhead_dynamics.pdf")
plt.close()

# ---------------------------------------------------------------- #

In [ ]:
from IPython.display import IFrame, display
print('Saved: fig_overhead_dynamics.pdf')


## Figure 3: Decision success rate vs mobility

In [11]:
# Figure 3: Decision success rate vs mobility (node speed)
# ---------------------------------------------------------------- #
speeds = np.linspace(2.0, 20.0, 8)
success_curves = {p: [] for p in POLICIES}
N_SEEDS_F3 = 4
for sp in speeds:
    for p in POLICIES:
        vals = []
        for s in range(N_SEEDS_F3):
            local_rng = np.random.default_rng(300 + s)
            res = run_policy(p, sp, event_mid, T=150, rng=local_rng)
            vals.append(res["decision_success_rate"] * 100)
        success_curves[p].append(np.mean(vals))

plt.figure(figsize=(5.2, 3.6))
for p in POLICIES:
    lw = 2.4 if p == "cdt" else 1.5
    plt.plot(speeds, success_curves[p], label=LABELS[p], color=COLORS[p],
              linestyle=STYLES[p], linewidth=lw, marker="o", markersize=3)
plt.xlabel("Node mobility (m/s)")
plt.ylabel("Decision success rate (%)")
plt.ylim(0, 105)
plt.legend(fontsize=6.5, loc="lower left")
plt.tight_layout()
plt.savefig("fig_success_mobility.pdf")
plt.close()

# ---------------------------------------------------------------- #

In [ ]:
from IPython.display import IFrame, display
print('Saved: fig_success_mobility.pdf')


## Summary table and Figure 2 raw data

In [ ]:
# Summary table at an operating point (moderate mobility + dynamics)
# ---------------------------------------------------------------- #
print(f"{'Policy':42s} {'NodeErr':>8s} {'Overhead':>9s} {'Success%':>9s} {'Latency':>8s}")
for p in POLICIES:
    metrics = {"avg_node_error": [], "avg_overhead": [], "decision_success_rate": [], "avg_latency": []}
    for s in range(6):
        local_rng = np.random.default_rng(400 + s)
        res = run_policy(p, speed_mid, event_mid, T=300, rng=local_rng)
        for k in metrics:
            metrics[k].append(res[k])
    print(f"{LABELS[p]:42s} {np.mean(metrics['avg_node_error']):8.3f} "
          f"{np.mean(metrics['avg_overhead']):9.1f} "
          f"{np.mean(metrics['decision_success_rate'])*100:9.1f} "
          f"{np.mean(metrics['avg_latency']):8.3f}")

print()
print("--- Figure 2 data: overhead to sustain 95% decision success vs event rate ---")
for i, er in enumerate(event_rates):
    row = ' '.join(f'{p}:{overhead_curves[p][i]:6.1f}' for p in POLICIES)
    print(f"event_rate={er:.4f}  {row}")